# Evaluación reproducible de LanguageBind preentrenado y adapter Helldivers

Este notebook es una versión reducida y reproducible del flujo principal. Su objetivo es evaluar recuperación vídeo-texto usando únicamente artefactos ya precalculados: índice de embeddings, ground truth manual y adapter entrenado. No se crean datasets, no se extraen embeddings nuevos y no se entrena ningún modelo.

La comparación central es entre dos configuraciones:

- **Fine-tuned / adaptada**: LanguageBind con el adapter lineal de Helldivers aplicado sobre los embeddings visuales.
- **Zero-shot / preentrenada**: LanguageBind tal como viene, sin adapter, usando el mismo índice y las mismas consultas.

Esta separación garantiza que cualquier diferencia en métricas procede de la adaptación al dominio, no de cambios en ventanas, ground truth o generación de datos.

## Artefactos requeridos

El notebook asume que los siguientes ficheros ya existen:

- `indexes/proxy_720p30_w15_s5/`: índice precalculado con embeddings de vídeo, audio y escena.
- `data/ground_truth/ground_truth_dataset.json`: consultas de evaluación con múltiples `window_index` relevantes por consulta.
- `data/models/helldivers_adapter.pth`: adapter lineal entrenado previamente.
- `scripts/evaluate_model.py`: script reproducible de evaluación con **Multi-Target Recall@K** y **Mean Reciprocal Rank (MRR)**.

El diseño evita pasos caros y no deterministas dentro del notebook. Esto lo hace más adecuado para revisión académica: el evaluador puede ejecutar la evaluación sin regenerar datos ni depender de decisiones manuales intermedias.

In [1]:
from pathlib import Path
import json
import sys

import torch
import torch.nn.functional as F

try:
    import pandas as pd
except ImportError:
    pd = None

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data" / "proxy_720p30.mp4").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))

from scripts.build_languagebind_index import default_index_dir, load_languagebind_index
from scripts.tfvtg_poc import (
    extract_text_embedding,
    load_languagebind_models,
    rank_windows,
    require_cuda,
    weighted_fuse_embeddings,
)
from scripts.train_linear_probe import VideoTextAdapter

VIDEO_PATH = REPO_ROOT / "data" / "proxy_720p30.mp4"
WINDOW_SECONDS = 15
STRIDE_SECONDS = 5
INDEX_DIR = REPO_ROOT / default_index_dir(VIDEO_PATH, WINDOW_SECONDS, STRIDE_SECONDS)
GROUND_TRUTH_PATH = REPO_ROOT / "data" / "ground_truth" / "ground_truth_dataset.json"
HELLDIVERS_ADAPTER_PATH = REPO_ROOT / "data" / "models" / "helldivers_adapter.pth"
CACHE_DIR = REPO_ROOT / "cache_dir"
EVALUATION_SCRIPT = REPO_ROOT / "scripts" / "evaluate_model.py"
ZERO_SHOT_ADAPTER_PATH = REPO_ROOT / "missing_zero_shot_adapter.pth"

TOP_K = 5
VIDEO_WEIGHT = 0.8
AUDIO_WEIGHT = 0.2

required_paths = {
    "index metadata": INDEX_DIR / "metadata.json",
    "ground truth": GROUND_TRUTH_PATH,
    "adapter": HELLDIVERS_ADAPTER_PATH,
    "evaluation script": EVALUATION_SCRIPT,
}

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required artifacts: {missing}")

print(f"Repo root: {REPO_ROOT}")
print(f"Index dir: {INDEX_DIR}")
print(f"Ground truth: {GROUND_TRUTH_PATH}")
print(f"Adapter: {HELLDIVERS_ADAPTER_PATH}")
print(f"Top-K: {TOP_K}")
print(f"Fusion weights: video={VIDEO_WEIGHT}, audio={AUDIO_WEIGHT}")

Repo root: /home/ruben/Documents/AINE/aine-highlights
Index dir: /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5
Ground truth: /home/ruben/Documents/AINE/aine-highlights/data/ground_truth/ground_truth_dataset.json
Adapter: /home/ruben/Documents/AINE/aine-highlights/data/models/helldivers_adapter.pth
Top-K: 5
Fusion weights: video=0.8, audio=0.2


/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga de índice y modelos

El índice ya contiene las ventanas temporales y los embeddings precalculados. Por tanto, esta sección solo los carga en memoria y prepara las ramas de LanguageBind necesarias para codificar consultas de texto.

La carga de modelos sigue siendo necesaria porque cada consulta textual debe convertirse en un embedding comparable con las ventanas del vídeo. Sin embargo, el vídeo y el audio no se vuelven a procesar: se reutilizan los tensores guardados en disco.

In [2]:
device = require_cuda()
print(f"Using device: {device}")

index = load_languagebind_index(INDEX_DIR)
with GROUND_TRUTH_PATH.open("r", encoding="utf-8") as handle:
    ground_truth = json.load(handle)

video_model, audio_model, _, _, text_tokenizer = load_languagebind_models(
    cache_dir=CACHE_DIR,
    device=device,
)

print(f"Windows: {len(index.windows)}")
print(f"Scene embeddings: {tuple(index.scene_embeddings.shape)}")
print(f"Ground-truth queries: {len(ground_truth)}")
print(f"Valid queries: {sum(1 for item in ground_truth if item['target_windows'])}")

Using device: cuda


/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Windows: 1243
Scene embeddings: (1243, 768)
Ground-truth queries: 16
Valid queries: 15


## Preparación de espacios de recuperación

Para comparar de forma justa, ambas configuraciones usan el mismo índice, las mismas ventanas y los mismos pesos de fusión multimodal:

- En **fine-tuned**, se aplica `VideoTextAdapter` a los embeddings visuales antes de fusionarlos con audio.
- En **zero-shot**, se usan los embeddings visuales originales de LanguageBind, sin ninguna adaptación.

La fusión usa **Late Fusion** con peso visual `0.8` y peso de audio `0.2`. Esta ponderación mantiene el foco en los eventos visuales del gameplay sin descartar señales acústicas como gritos, risas o explosiones.

In [3]:
def build_scene_embeddings(adapter_path: Path | None):
    video_embeddings = index.video_embeddings.to(device=device, dtype=torch.float32)
    audio_embeddings = index.audio_embeddings.to(device=device, dtype=torch.float32)

    if adapter_path is not None:
        embedding_dim = int(video_embeddings.shape[-1])
        adapter = VideoTextAdapter(embedding_dim=embedding_dim).to(device)
        adapter.load_state_dict(torch.load(adapter_path, map_location=device))
        adapter.eval()
        with torch.no_grad():
            video_embeddings = F.normalize(adapter(video_embeddings), dim=-1)
    else:
        video_embeddings = F.normalize(video_embeddings, dim=-1)

    audio_embeddings = F.normalize(audio_embeddings, dim=-1)
    return weighted_fuse_embeddings(
        video_embeddings,
        audio_embeddings,
        video_weight=VIDEO_WEIGHT,
        audio_weight=AUDIO_WEIGHT,
    )

with torch.no_grad():
    fine_tuned_scene_embeddings = build_scene_embeddings(HELLDIVERS_ADAPTER_PATH)
    zero_shot_scene_embeddings = build_scene_embeddings(None)

print(f"Fine-tuned embeddings: {tuple(fine_tuned_scene_embeddings.shape)}")
print(f"Zero-shot embeddings: {tuple(zero_shot_scene_embeddings.shape)}")

Fine-tuned embeddings: (1243, 768)
Zero-shot embeddings: (1243, 768)


## Top-K y métricas manuales

Esta sección calcula dos niveles de evidencia:

- **Top-K por consulta**: permite inspeccionar qué ventanas devuelve cada configuración y si alguna coincide con los `target_windows` esperados.
- **Métricas agregadas**: resumen el rendimiento con **Recall@1**, **Recall@3**, **Recall@5** y **MRR**.

Como cada consulta puede tener múltiples ventanas correctas, el Recall@K se considera positivo si cualquiera de las ventanas relevantes aparece entre los primeros `K` resultados.

In [4]:
def calculate_metrics(results, target_windows, top_k_max=5):
    retrieved_indices = [result.window_index for result in results]

    mrr = 0.0
    for rank, window_index in enumerate(retrieved_indices, start=1):
        if window_index in target_windows:
            mrr = 1.0 / rank
            break

    recalls = {}
    for k in [1, 3, 5]:
        if k <= top_k_max:
            recalls[f"R@{k}"] = float(any(
                window_index in target_windows for window_index in retrieved_indices[:k]
            ))
    return mrr, recalls


def retrieve(query, scene_embeddings):
    text_embedding = extract_text_embedding(
        model=video_model,
        tokenizer=text_tokenizer,
        query=query,
        device=device,
    )
    return rank_windows(
        windows=index.windows,
        scene_embeddings=scene_embeddings,
        text_embedding=text_embedding,
        top_k=TOP_K,
    )


def evaluate_setting(setting_name, scene_embeddings):
    metric_rows = []
    topk_rows = []

    with torch.no_grad():
        for item in ground_truth:
            query = item["query"]
            targets = set(item["target_windows"])
            if not targets:
                continue

            results = retrieve(query, scene_embeddings)
            mrr, recalls = calculate_metrics(results, targets, TOP_K)
            metric_rows.append({
                "setting": setting_name,
                "query": query,
                "target_windows": sorted(targets),
                "MRR": mrr,
                **recalls,
            })

            for result in results:
                topk_rows.append({
                    "setting": setting_name,
                    "query": query,
                    "rank": result.rank,
                    "window_index": result.window_index,
                    "is_target": result.window_index in targets,
                    "score": result.score,
                    "start_s": result.start_s,
                    "end_s": result.end_s,
                    "target_windows": sorted(targets),
                })

    return metric_rows, topk_rows

fine_metrics, fine_topk = evaluate_setting("fine_tuned", fine_tuned_scene_embeddings)
zero_metrics, zero_topk = evaluate_setting("zero_shot", zero_shot_scene_embeddings)

metric_rows = fine_metrics + zero_metrics
topk_rows = fine_topk + zero_topk

if pd is not None:
    metrics_df = pd.DataFrame(metric_rows)
    topk_df = pd.DataFrame(topk_rows)
    summary_df = metrics_df.groupby("setting")[["MRR", "R@1", "R@3", "R@5"]].mean().reset_index()
    display(summary_df)
    display(topk_df)
else:
    for setting in ["fine_tuned", "zero_shot"]:
        rows = [row for row in metric_rows if row["setting"] == setting]
        print(setting)
        print({
            metric: sum(row[metric] for row in rows) / len(rows)
            for metric in ["MRR", "R@1", "R@3", "R@5"]
        })
        print()
    for row in topk_rows:
        print(row)

fine_tuned
{'MRR': 0.4722222222222222, 'R@1': 0.4, 'R@3': 0.5333333333333333, 'R@5': 0.6}

zero_shot
{'MRR': 0.03333333333333333, 'R@1': 0.0, 'R@3': 0.06666666666666667, 'R@5': 0.06666666666666667}

{'setting': 'fine_tuned', 'query': 'A player is accidentally killed by friendly fire.', 'rank': 1, 'window_index': 217, 'is_target': False, 'score': 0.30299872159957886, 'start_s': 1085.0, 'end_s': 1100.0, 'target_windows': [23, 211, 551, 558, 1029, 1223]}
{'setting': 'fine_tuned', 'query': 'A player is accidentally killed by friendly fire.', 'rank': 2, 'window_index': 194, 'is_target': False, 'score': 0.28363949060440063, 'start_s': 970.0, 'end_s': 985.0, 'target_windows': [23, 211, 551, 558, 1029, 1223]}
{'setting': 'fine_tuned', 'query': 'A player is accidentally killed by friendly fire.', 'rank': 3, 'window_index': 392, 'is_target': False, 'score': 0.28213271498680115, 'start_s': 1960.0, 'end_s': 1975.0, 'target_windows': [23, 211, 551, 558, 1029, 1223]}
{'setting': 'fine_tuned', 'query

## Evaluación reproducible mediante script

La celda anterior permite inspección detallada dentro del notebook. Para reproducibilidad operativa, también se ejecuta `scripts/evaluate_model.py`, que calcula las mismas métricas desde línea de comandos.

La primera llamada evalúa el adapter fine-tuned. La segunda fuerza el caso **zero-shot** usando una ruta de adapter inexistente, de modo que el script conserva los embeddings base de LanguageBind.

In [5]:
print("Fine-tuned evaluation")
!{sys.executable} {EVALUATION_SCRIPT} --index-dir {INDEX_DIR} --mode manual --ground-truth {GROUND_TRUTH_PATH} --adapter {HELLDIVERS_ADAPTER_PATH} --top-k {TOP_K} --video-weight {VIDEO_WEIGHT} --audio-weight {AUDIO_WEIGHT}

print("Zero-shot evaluation")
!{sys.executable} {EVALUATION_SCRIPT} --index-dir {INDEX_DIR} --mode manual --ground-truth {GROUND_TRUTH_PATH} --adapter {ZERO_SHOT_ADAPTER_PATH} --top-k {TOP_K} --video-weight {VIDEO_WEIGHT} --audio-weight {AUDIO_WEIGHT}

Fine-tuned evaluation
Loading Index from /home/ruben/Documents/AINE/aine-highlights/indexes/proxy_720p30_w15_s5...
Mode: MANUAL Ground Truth (Multi-Target from /home/ruben/Documents/AINE/aine-highlights/data/ground_truth/ground_truth_dataset.json)
Loading Models...
/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/home/ruben/Documents/AINE/aine-highlights/venv/lib/python3.12/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
Applying Adapter from /home/ruben/Documents/AINE/aine-highlights/data/mode